# Frontier Models, Untrained

`price_baselines` established what a model has to beat: guess the training-set
average and you are off by a fixed amount, with an r² of exactly 0%. Anything
that has actually learned something scores better than that.

This notebook hands the same held-out items to three frontier models and asks
them, in plain English, what the product costs. None of them has been trained on
this dataset. They have never seen a price from it. All they get is the product
text and one instruction.

| Model | OpenRouter ID |
|---|---|
| Claude | `anthropic/claude-sonnet-5` |
| GPT | `openai/gpt-5.2` |
| Gemini | `google/gemini-3.1-pro-preview` |

The point is not to crown a winner. It is to find out how much of this task is
already solved by general world knowledge -- a model that has read the internet
knows roughly what a guitar pedal costs -- and to set the number that
fine-tuning will later have to beat. **Lower error is better.**

This one costs real money: roughly 60 cents for a full run at `SIZE = 100`, most
of it Gemini's. `SIZE` is the knob if you want to spend less or measure more
precisely.

## Imports

In [ ]:
import pandas as pd
from dotenv import load_dotenv

from notebooks.helpers.evaluator import evaluate
from notebooks.helpers.frontier import FrontierPricer, Model
from notebooks.models.items import Item

load_dotenv(override=True)

## Load the data

The same three splits as the baselines notebook, and the same rule: the models
are scored on `test`, which nothing has been trained on. Here that rule costs
nothing to keep -- none of these models has been trained on any of it.

Note `summary` is empty on `test`. It was written for the training split only,
so the prompt has to be built from `full`, the raw product text.

In [ ]:
DATASET = "ed-donner/items_raw_lite"

train, validation, test = Item.from_hub(DATASET)

print(f"train={len(train):,} validation={len(validation):,} test={len(test):,}")
print(test[0])

## The bar to beat

Guessing the training-set average is the most you can do without looking at the
product at all. Re-scored here so the number sits on this page rather than in
another notebook, and so the leaderboard at the end has an honest first row.

No API calls, so this cell is free and instant.

In [ ]:
SIZE = 100

training_average = sum(item.price for item in train) / len(train)
print(f"training average = ${training_average:,.2f}")


def constant_pricer(item):
    return training_average


constant = evaluate(constant_pricer, test, size=SIZE)

## What the model actually sees

Before spending anything, look at the request. A system message stating the job,
a user message carrying the product text, and nothing else -- no examples, no
price ranges, no category hints. That is what makes this zero-shot.

In [ ]:
example = test[0]

for message in FrontierPricer(Model.Claude).messages(example):
    print(f"--- {message['role']} ---")
    print(message["content"][:600])
    print()

print(f"--- truth: ${example.price:.2f} ---")

## Smoke test

Three items, three models, raw replies. Run this before the paid run -- it costs
a fraction of a cent and catches the failure that would otherwise be invisible.

If a reply comes back empty, `evaluate` scores it as a guess of `$0` and the
model looks catastrophically bad on the chart when it was merely truncated.
Seeing the strings here rules that out.

In [ ]:
for model in Model:
    pricer = FrontierPricer(model)
    print(f"--- {model.name} ({model.value}) ---")
    for item in test[:3]:
        print(f"  ${item.price:>8,.2f} truth | {pricer(item)!r}")

## Score all three

`evaluate` takes any callable of one `Item`, and `FrontierPricer` is exactly
that -- it returns the model's raw reply, and the harness pulls the number out
of the string. So these three runs are scored by the same code, on the same
items, as the linear regression in the baselines notebook.

Each run makes `SIZE` API calls across five threads. Expect a couple of minutes
per model.

`blanks` counts replies that carried no number at all. Those score as `$0` and
drag the average down, so a non-zero count is worth knowing about before you
read anything into the result.

In [ ]:
pricers = {model: FrontierPricer(model) for model in Model}
testers = {}

for model, pricer in pricers.items():
    testers[model] = evaluate(pricer, test, size=SIZE, title=model.name)
    if pricer.blanks:
        print(f"{model.name}: {len(pricer.blanks)} of {SIZE} replies carried no number")

## Leaderboard

Average absolute error, in dollars, over the same `SIZE` items. `vs constant` is
how much of the gap between guessing-the-average and being-exactly-right each
model closed -- the same thing r² measures, stated as a percentage of the
baseline's error.

In [ ]:
def average_error(tester):
    return sum(result.error for result in tester.results) / len(tester.results)


baseline = average_error(constant)

frame = pd.DataFrame(
    [{"predictor": "constant (train mean)", "error": baseline}]
    + [
        {"predictor": model.name, "error": average_error(tester)}
        for model, tester in testers.items()
    ]
)
frame["vs constant"] = (1 - frame["error"] / baseline).map("{:.1%}".format)
frame["error"] = frame["error"].map("${:,.2f}".format)
frame.sort_values("error").reset_index(drop=True)

## What this tells you

A frontier model doing well here is not evidence that it learned this dataset.
It is evidence that pricing a consumer product is largely a world-knowledge
task, and these models have a lot of world knowledge. The interesting number is
the gap that remains: whatever a model cannot get from general knowledge is what
training on the training split has to supply.

Two things to watch on the charts:

- **The confidence band.** At `SIZE = 100` it is wide. Two models a few dollars
  apart are not distinguishable -- raise `SIZE` before believing a ranking.
- **The scatter.** These models tend to round to familiar price points, so the
  dots cluster in horizontal lines at $19.99, $49.99, $99. A fine-tuned model
  spreads out.

## Next

Fine-tune these models on `train` and score them here again, unchanged. Nothing
in this notebook needs to know a model was trained -- swap the OpenRouter ID in
`Model` for the fine-tuned one and the comparison holds.

Knobs, same as the baselines notebook:

```python
evaluate(pricer, test, size=500)      # tighter confidence band, five times the cost
evaluate(pricer, test, workers=20)    # more threads; watch for rate limits
worst = sorted(testers[Model.GPT].results, key=lambda result: -result.error)[:10]
```